# 4. Coding Agents

**기본 경로는 자기 AI 도구로 요청하고 결과 파일을 실제로 완성하는 실습입니다.** 아래 대체 fixture는 AI 계정이나 도구 접근이 없을 때만 씁니다. 고정 fixture의 출력은 실제 모델 응답이나 실제 도구 호출 기록이 아닙니다.

Python 문법 구현은 요구하지 않습니다. 제공된 검사 셀은 그대로 실행하고 결과를 읽으면 됩니다.

## 기본 경로 A. coding agent의파일작업

3강의작업폴더를실제codingagent에서여세요. 파일읽기,수정,명령실행을지원하는세션을선택합니다.

**읽기첫요청**

> Read README.md, inspection.csv, checks.py and app.py. Explain the outputs and run command. List proposed edits for a quality-status column. Do not edit or run commands yet.

**작업요청**

> Improve inspection-report.html with a quality-status column. Preserve JSON values, all rows and their order. Edit app.py only. Do not change CSV or checker. Run the generator and checks.py. Return actual artifact paths and command output.

도구호출을펼쳐read/edit/run/test를확인하고반환된HTML을직접여세요. 실제변경diff와완료물경로를남깁니다.

## 기본 경로 B. 공개읽기MCP연결

공식문서: https://docs.devin.ai/work-with-devin/deepwiki-mcp

1. MCP를지원하는client의server설정을여세요. VSCode에서는CommandPalette의`MCP: Add Server`를쓸수있습니다.
2. HTTP를선택하고`https://mcp.deepwiki.com/mcp`를넣습니다. 이름은deepwiki로구분하세요.
3. publisher,endpoint,요청한권한을확인하세요. 공개repository이름만보내며사적인프로젝트파일은첨부하지않습니다.
4. 실제도구목록과schema를여세요. 공식문서에는read_wiki_structure,read_wiki_contents,ask_question이안내되어있습니다. 관찰한목록을기록하세요.
5. 아래요청을보내실제도구호출을펼치고응답을확인하세요.

> Use deepwiki MCP read_wiki_structure for the public repository python/cpython. Use the actual tool schema. Do not change files. Report the tool name and returned topics.

6. 반환문서에서CSV parsing관련내용을찾도록후속질문하고source를확인하세요. 서버가해당repository를지원하지않거나연결이실패하면실제오류와원인을기록하고완료라고표시하지않습니다.

이notebook은원격호출을자동실행하거나연결성공을주장하지않습니다.

## 기본 경로의과제

- README사용법초안만별도작업에위임하고기존명령과맞는지본뒤합치세요. 원본파일을바로바꾸게하지말고텍스트초안을받습니다.
- localedit와공개문서read의실제권한범위를각각설명하세요.

제출물은개선된보고서,실제도구기록,diff/checkpoint증거,MCP연결과한번호출기록입니다.

## 대체 경로 C. 도구접근이없는경우

아래는사람이작성한fixedrecord와reference보고서입니다. 실제agent나MCP호출로소개하지마세요. 외부호출없이산출물과기록의대응을연습합니다.

In [1]:
from pathlib import Path
import tempfile,importlib.util,json,html,shutil
starter=next((p for p in [Path("labs/3_starter"),Path("3_starter")]if p.exists()),None)
if starter is None:raise FileNotFoundError("Unzip 3_starter.zip beside this notebook, or run from the course root")
spec=importlib.util.spec_from_file_location("fixture_checks",starter/"checks.py")
checker=importlib.util.module_from_spec(spec);spec.loader.exec_module(checker)
reference=checker.expected(starter/"inspection.csv")
assert reference["summary"]=={"raw":8,"valid":6,"flagged":4,"missing":1,"invalid":1}
print("Source-derived reference:",reference["summary"])


Source-derived reference: {'raw': 8, 'valid': 6, 'flagged': 4, 'missing': 1, 'invalid': 1}


In [2]:
with tempfile.TemporaryDirectory(prefix="ax-report-check-")as td:
    target=Path(td)
    shutil.copyfile(starter/"inspection.csv",target/"inspection.csv")
    (target/"report.json").write_text(json.dumps(reference),encoding="utf-8")
    head="<h1>Human-authored reference fixture</h1>"+html.escape(str(reference["summary"]))
    rows="<table><tr><th>ID</th><th>Raw score</th><th>Status</th></tr>"
    for row in reference["rows"]:
        rows+="<tr>"+"".join("<td>"+html.escape(str(row[k]))+"</td>"for k in ["id","raw_score","status"])+"</tr>"
    (target/"inspection-report.html").write_text(head+rows+"</table>",encoding="utf-8")
    print("Reference fixture checker:",checker.check(target))
    # Meaningful negative case: dropping a duplicate row must fail.
    broken=dict(reference);broken["rows"]=reference["rows"][:-1]
    (target/"report.json").write_text(json.dumps(broken),encoding="utf-8")
    try:checker.check(target)
    except AssertionError:print("A dropped-row candidate is rejected")
    else:raise AssertionError("Checker accepted a dropped row")


Reference fixture checker: {'raw': 8, 'valid': 6, 'flagged': 4, 'missing': 1, 'invalid': 1}
A dropped-row candidate is rejected


In [3]:
documented_tools=["read_wiki_structure","read_wiki_contents","ask_question"]
fixture_record={"kind":"supplied simulation, not a live tool call",
                "server":"deepwiki","tool":"read_wiki_structure",
                "repository":"python/cpython","returned_topics":None,"executed":False}
assert fixture_record["tool"]in documented_tools
assert not fixture_record["executed"] and fixture_record["returned_topics"]is None
print(fixture_record)

{'kind': 'supplied simulation, not a live tool call', 'server': 'deepwiki', 'tool': 'read_wiki_structure', 'repository': 'python/cpython', 'returned_topics': None, 'executed': False}


### 대체경로의과제와해답

- “연결됨”이라는문장만으로실제도구호출을했다고판정할수있는지설명하세요.
- 실행한record와실행하지않은record를구분하는검사를고르세요.

답은실제client의호출상세에서server,tool,입력,반환값을대조하는것입니다. 아래검사는이fixture가실행기록이아님을확인할뿐원격연결을검증하지않습니다.

In [4]:
has_execution_evidence=bool(fixture_record["executed"]and fixture_record["returned_topics"]is not None)
assert not has_execution_evidence
print("No live execution evidence is present in the supplied fixture")

No live execution evidence is present in the supplied fixture


## 구현참고

agentharness의상태기계나Python도구dispatcher구현은이번필수과제가아닙니다. 먼저실제도구로read/edit/run/test와MCP연결을사용하고,내부코드는프로그래밍지식이생긴뒤읽습니다.